# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import anthropic
import gradio as gr
import google.generativeai
from pathlib import Path
from dataclasses import dataclass
from pydantic import BaseModel

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

In [ ]:
openai = OpenAI()
claude = anthropic.Anthropic()

In [ ]:
google.generativeai.configure()

In [ ]:
MODEL = "gpt-4o-mini"

In [ ]:
system_message = "You are a helpful assistant for an Airline called FlightAI. "
system_message += "Give short, courteous answers, no more than 1 sentence. "
system_message += "Always be accurate. If you don't know the answer, say so."

In [ ]:
def translator(lang, text):
    gemini = google.generativeai.GenerativeModel(
            model_name='gemini-2.0-flash',
            system_instruction=(
                f"You are the professional translater from English to {lang}."
                "Your response should only contains the translation for a given text.."
            )
    )
    response = gemini.generate_content(text)
    return response.text

In [ ]:
print(translator("Turkish", "Could you recommand some flight to London?"))

In [ ]:
@dataclass
class BookTicket():
    first_name: str
    surname: str
    flight: str
    price: float

In [ ]:
class BookedTicket(BaseModel):
    first_name: str
    surname: str
    flight: str
    price: float

In [ ]:
def book_ticket(first_name, surname, flight, price):
    rec = BookedTicket(first_name=first_name, surname=surname, flight=flight, price=price)
    with open((Path.cwd() / 'booked_tickets_registry.txt'), mode= 'at') as fh:
        fh.write(f"{rec.model_dump_json()}\n")
    return rec.model_dump()

In [ ]:
book_ticket('t','t','t',2.5)

In [ ]:
# !more booked_tickets_registry.txt

In [ ]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool get_ticket_price called for {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")

In [ ]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
book_ticket_function = {
    "name": "book_ticket",
    "description": ("Take the customer data like first name and surname and book the tikcet for the selected flight and for a accepted price. "
                    "Call this whenever you need to book a customer ask you to book ticket."
                   ),
    "parameters": {
        "type": "object",
        "properties": {
            "first_name": {
                "type": "string",
                "description": "The first name of the customer who is booking ticket",
            },
            "surname": {
                "type": "string",
                "description": "The surnuma of the customer who is booking ticket",
            },
            "flight": {
                "type": "string",
                "description": "The name of the selected flight",
            },
            "price": {
                "type": "number",
                "description": "The price of the selected flight",
            },
        },
        "required": ["first_name", "surname", "flight", "price"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": book_ticket_function},
]

In [ ]:
import base64
from io import BytesIO
from PIL import Image

In [ ]:
def artist(city):
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
import base64
from io import BytesIO
from PIL import Image
from IPython.display import Audio, display

def talker(message):
    response = openai.audio.speech.create(
        model="tts-1",
        voice="onyx",
        input=message)

    audio_stream = BytesIO(response.content)
    output_filename = "output_audio.mp3"
    with open(output_filename, "wb") as f:
        f.write(audio_stream.read())

    # Play the generated audio
    display(Audio(output_filename, autoplay=True))

talker("Well, hi there")

In [ ]:
def chat1(history, transaltion, lang, history_translation):
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    # image = None
    
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response, city = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        # image = artist(city)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
        
    # reply = response.choices[0].message.content
    response = response.choices[0].message.content
    history += [{"role":"assistant", "content":response}]
    if transaltion:
        transalted_message = translator(lang, response)
        history_translation += [{"role":"assistant", "content":transalted_message}]

    # Comment out or delete the next line if you'd rather skip Audio for now..
    # talker(reply)
    
    # return history, image
    print(chatbot_translation)
    return history, history_translation

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    print(f"MLL response:\n{response.choices[0].message}")
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response, city = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    name = tool_call.function.name
    city = None
    arguments = json.loads(tool_call.function.arguments)
    if name == "get_ticket_price":
        city = arguments.get('destination_city')
        # max_price = arguments.get('max_price')
        price = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": json.dumps({"destination_city": city,"price": price}),
            "tool_call_id": tool_call.id
        }
    elif name == "book_ticket":
        first_name = arguments.get('first_name')
        surname = arguments.get('surname')
        flight = arguments.get('flight')
        price = arguments.get('price')
        ticket = book_ticket(first_name, surname, flight, price)
        response = {
            "role": "tool",
            "content": json.dumps(ticket),
            "tool_call_id": tool_call.id
        }
        city = flight
    else:
        raise ValueError(f"Unknown tool name {name}")
    return response, city

In [ ]:
# gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
# help(gr.Checkbox)

In [ ]:
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        chatbot_translation = gr.Chatbot(height=500, type="messages")
        # image_output = gr.Image(height=500)
    with gr.Row():
        entry = gr.Textbox(label="Chat with our AI Assistant:")
    with gr.Row():
        clear = gr.Button("Clear")
        transaltion = gr.Checkbox(value=False, label="Translation to:")
        lang = gr.Textbox(label="Language:")

    def do_entry(message, transaltion, lang, history, history_transaltion):
        history += [{"role":"user", "content":message}]
        # print(f"history:{history}")
        # print(f"history_transaltion:{history_transaltion}")
        if transaltion:
            transalted_message = translator(lang, message)
            history_transaltion += [{"role":"user", "content":transalted_message}]
            # print(f"transaltion - history_transaltion:{history_transaltion}")
        return "", history, history_transaltion

    def toggle_textbox(enable):
        if enable:
            return gr.update(interactive=True, placeholder="Możesz pisać...")
        else:
            return gr.update(interactive=False, placeholder="Pole wyłączone", value="")

    transaltion.change(fn=toggle_textbox, inputs=transaltion, outputs=lang)
    entry.submit(do_entry, inputs=[entry, transaltion, lang, chatbot, chatbot_translation], outputs=[entry, chatbot, chatbot_translation]).then(
        chat1, inputs=[chatbot, transaltion, lang, chatbot_translation] , outputs=[chatbot, chatbot_translation])
    clear.click(lambda: (None, None), inputs=None, outputs=[chatbot, chatbot_translation], queue=False)

ui.launch(inbrowser=True)

In [ ]:
print(gr.__version__)